# DeepONet: Learning the Anti-Derivative Operator

**Webinar: Scientific Machine Learning & DeepONet**  
**NCSA — Diab Abueidda & Qibang Liu**

---

## Problem Statement

We want to learn the **anti-derivative operator**:

$$G(u)(y) = \int_0^y u(t)\, dt, \quad y \in [0, 1]$$

Given an input function $u(x)$, the operator maps it to its anti-derivative $s(y)$.

### DeepONet Setup
- **Branch input**: $u$ evaluated at $m$ fixed sensor locations $\rightarrow [u(x_1), u(x_2), \ldots, u(x_m)]$
- **Trunk input**: query point $y$ where we evaluate the output
- **Output**: $G(u)(y) = \int_0^y u(t)\, dt$

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Generation

We generate input functions $u(x)$ as a sum of two sine terms with random coefficients:

$$u(x) = a_1 \sin(\omega_1 \pi x) + a_2 \sin(\omega_2 \pi x)$$

where $a_1, a_2 \sim \mathcal{U}[-2, 2]$ and $\omega_1, \omega_2 \sim \mathcal{U}[1, 6]$.

The **analytical** anti-derivative is:

$$G(u)(y) = \int_0^y u(t)\,dt = \frac{a_1}{\omega_1 \pi}\left[1 - \cos(\omega_1 \pi y)\right] + \frac{a_2}{\omega_2 \pi}\left[1 - \cos(\omega_2 \pi y)\right]$$

Having an analytical expression means we have **exact ground truth** — no numerical approximation needed.

In [ ]:
def generate_sine_samples(n_samples, x_grid,
                          amp_range=(-2, 2), freq_range=(1, 6)):
    """
    Generate random functions as a sum of two sine terms:
        u(x) = a1 * sin(w1 * pi * x) + a2 * sin(w2 * pi * x)
    """
    a1 = np.random.uniform(*amp_range, size=n_samples)
    a2 = np.random.uniform(*amp_range, size=n_samples)
    w1 = np.random.uniform(*freq_range, size=n_samples)
    w2 = np.random.uniform(*freq_range, size=n_samples)
    
    u_samples = (a1[:, None] * np.sin(w1[:, None] * np.pi * x_grid[None, :]) +
                 a2[:, None] * np.sin(w2[:, None] * np.pi * x_grid[None, :]))
    
    params = np.stack([a1, w1, a2, w2], axis=1)
    return u_samples, params


def antiderivative_analytical(y_grid, params):
    """
    Compute the ANALYTICAL anti-derivative:
        s(y) = a1/(w1*pi) * [1 - cos(w1*pi*y)] + a2/(w2*pi) * [1 - cos(w2*pi*y)]
    """
    a1 = params[:, 0:1]
    w1 = params[:, 1:2]
    a2 = params[:, 2:3]
    w2 = params[:, 3:4]
    
    y = y_grid[None, :]
    
    s = (a1 / (w1 * np.pi) * (1 - np.cos(w1 * np.pi * y)) +
         a2 / (w2 * np.pi) * (1 - np.cos(w2 * np.pi * y)))
    
    return s

In [ ]:
# --- Grid definitions ---
m = 100        # Sensor points (branch input)
x_sensors = np.linspace(0, 1, m)

n_query = 80   # Query points (trunk input) — DIFFERENT from sensors
y_query_np = np.linspace(0, 1, n_query)

print(f'Sensor grid: {m} points (branch input)')
print(f'Query grid:  {n_query} points (trunk input)')
print(f'Sensors ≠ Query points — this is a key feature of DeepONet!\n')

# --- Generate data ---
N_train = 1000
N_test = 200

# Generate u directly at sensor locations (branch input)
u_train, params_train = generate_sine_samples(N_train, x_sensors)
u_test, params_test = generate_sine_samples(N_test, x_sensors)

# Compute s directly at query locations (analytical ground truth)
s_train = antiderivative_analytical(y_query_np, params_train)
s_test = antiderivative_analytical(y_query_np, params_test)

print(f'u_train shape: {u_train.shape}  (N_train × m_sensors)')
print(f's_train shape: {s_train.shape}  (N_train × n_query)')

# Show a few example parameterizations
print(f'\nExample functions generated:')
for i in range(3):
    a1, w1, a2, w2 = params_train[i]
    print(f'  u_{i}(x) = {a1:+.2f}·sin({w1:.2f}πx) {a2:+.2f}·sin({w2:.2f}πx)')

### Visualize the data

A few sample input functions $u(x)$ evaluated at sensor locations, and their analytical anti-derivatives $s(y)$ evaluated at query locations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for i in range(5):
    # Plot u at sensor locations
    axes[0].plot(x_sensors, u_train[i], alpha=0.8)
    # Plot s at query locations
    axes[1].plot(y_query_np, s_train[i], alpha=0.8)

axes[0].set_title('Input functions $u(x) = a_1 \\sin(\\omega_1 \\pi x) + a_2 \\sin(\\omega_2 \\pi x)$', fontsize=12)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$u(x)$')
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Anti-derivatives $s(y)$ (analytical)', fontsize=12)
axes[1].set_xlabel('$y$')
axes[1].set_ylabel('$s(y)$')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. DeepONet Architecture

We build the **DeepONet** with:
- **Branch net** (MLP): takes $[u(x_1), \ldots, u(x_m)]$ → outputs coefficients $[b_1, \ldots, b_p]$
- **Trunk net** (MLP): takes query point $y$ → outputs basis functions $[t_1, \ldots, t_p]$
- **Output**: $G(u)(y) \approx \sum_{k=1}^{p} b_k(u) \cdot t_k(y) + b_0$

We use the **Cartesian product** format: for each input function, we evaluate at all query points simultaneously.

In [ ]:
class MLP(nn.Module):
    """Simple Multi-Layer Perceptron."""
    def __init__(self, layer_sizes, activation=nn.ReLU):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:  # No activation after last layer
                layers.append(activation())
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class DeepONet(nn.Module):
    """
    Deep Operator Network (Unstacked, Cartesian Product).
    
    Args:
        branch_layers: List of layer sizes for the branch net
        trunk_layers: List of layer sizes for the trunk net
    
    Note: The last layer of both branch and trunk must have the same width (p).
    """
    def __init__(self, branch_layers, trunk_layers):
        super().__init__()
        self.branch_net = MLP(branch_layers)
        self.trunk_net = MLP(trunk_layers)
        # Learnable bias
        self.bias = nn.Parameter(torch.zeros(1), requires_grad=True)
    
    def forward(self, u_sensors, y_query):
        """
        Forward pass.
        
        Args:
            u_sensors: Branch input, shape (batch_size, m)
            y_query:   Trunk input, shape (n_query, 1)
        
        Returns:
            output: shape (batch_size, n_query)
        """
        # Branch: (batch_size, m) -> (batch_size, p)
        b = self.branch_net(u_sensors)
        
        # Trunk: (n_query, 1) -> (n_query, p)
        t = self.trunk_net(y_query)
        
        # Dot product: (batch_size, p) x (n_query, p)^T -> (batch_size, n_query)
        # Using einsum for clarity
        output = torch.einsum('bp, np -> bn', b, t) + self.bias
        
        return output

In [ ]:
# --- Define the model ---
p = 40  # Width of the last layer (number of basis functions / coefficients)

branch_layers = [m, 128, 128, 128, p]   # Input: m sensor values
trunk_layers  = [1, 128, 128, 128, p]   # Input: 1 query coordinate (y)

model = DeepONet(branch_layers, trunk_layers).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Model has {n_params:,} trainable parameters')
print(model)

## 4. Prepare Data for Training

We use the **Cartesian product** format:
- Branch input: `u_train` of shape `(N_train, m)` — each function at **sensor** locations
- Trunk input: `y_query` of shape `(n_query, 1)` — the **query** locations (different from sensors!)
- Target: `s_train` of shape `(N_train, n_query)` — the analytical anti-derivative at each query point

**Key point**: the branch reads $u$ at $m=100$ sensor points, but the trunk evaluates the output at $n=80$ different query points.

In [ ]:
# Convert data to PyTorch tensors
u_train_tensor = torch.tensor(u_train, dtype=torch.float32).to(device)
s_train_tensor = torch.tensor(s_train, dtype=torch.float32).to(device)

u_test_tensor = torch.tensor(u_test, dtype=torch.float32).to(device)
s_test_tensor = torch.tensor(s_test, dtype=torch.float32).to(device)

# Query points for the trunk
y_query = torch.tensor(y_query_np.reshape(-1, 1), dtype=torch.float32).to(device)

print(f'Branch input (u at sensors): {u_train_tensor.shape}')
print(f'Trunk input  (query points): {y_query.shape}')
print(f'Target       (s at queries): {s_train_tensor.shape}')

## 5. Training Loop

In [ ]:
# --- Training configuration ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)
n_epochs = 20000
batch_size = 256

# Create a simple data loader
dataset = torch.utils.data.TensorDataset(u_train_tensor, s_train_tensor)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Loss function
loss_fn = nn.MSELoss()

In [ ]:
# --- Train the model ---
train_losses = []
test_losses = []

print('Training DeepONet...')
print('-' * 50)

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for u_batch, s_batch in dataloader:
        optimizer.zero_grad()
        
        # Forward pass
        s_pred = model(u_batch, y_query)
        
        # Compute loss
        loss = loss_fn(s_pred, s_batch)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    scheduler.step()
    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)
    
    # Evaluate on test set periodically
    if (epoch + 1) % 500 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            s_test_pred = model(u_test_tensor, y_query)
            test_loss = loss_fn(s_test_pred, s_test_tensor).item()
        test_losses.append(test_loss)
        print(f'Epoch {epoch+1:5d}/{n_epochs} | '
              f'Train Loss: {avg_train_loss:.6e} | '
              f'Test Loss: {test_loss:.6e}')

print('-' * 50)
print('Training complete!')

## 6. Results and Visualization

In [ ]:
# --- Plot training loss ---
plt.figure(figsize=(8, 4))
plt.semilogy(train_losses, alpha=0.7, label='Train Loss')
test_epochs = [0] + list(range(499, n_epochs, 500))
plt.semilogy(test_epochs, test_losses, 'o-', color='red', markersize=3, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Test Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Predict on test set ---
model.eval()
with torch.no_grad():
    s_test_pred = model(u_test_tensor, y_query).cpu().numpy()

# Compute relative L2 error for each test sample
rel_l2_errors = np.linalg.norm(s_test_pred - s_test, axis=1) / np.linalg.norm(s_test, axis=1)
print(f'Mean relative L2 error: {np.mean(rel_l2_errors):.4e}')
print(f'Max  relative L2 error: {np.max(rel_l2_errors):.4e}')

In [ ]:
# --- Visualize predictions vs analytical ground truth ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

test_indices = [0, 50, 150]

for col, idx in enumerate(test_indices):
    # Top row: input function u(x)
    axes[0, col].plot(x_sensors, u_test[idx], color='steelblue', linewidth=2)
    a1, w1, a2, w2 = params_test[idx]
    axes[0, col].set_title(f'$u = {a1:+.1f}\\sin({w1:.1f}\\pi x){a2:+.1f}\\sin({w2:.1f}\\pi x)$', fontsize=11)
    axes[0, col].set_xlabel('$x$')
    axes[0, col].set_ylabel('$u(x)$')
    axes[0, col].grid(True, alpha=0.3)
    
    # Bottom row: DeepONet vs analytical
    axes[1, col].plot(y_query_np, s_test[idx], 'k-', linewidth=2, label='Analytical')
    axes[1, col].plot(y_query_np, s_test_pred[idx], 'r--', linewidth=2, label='DeepONet')
    axes[1, col].set_title(f'Rel. $L^2$ error: {rel_l2_errors[idx]:.2e}', fontsize=12)
    axes[1, col].set_xlabel('$y$')
    axes[1, col].set_ylabel('$s(y)$')
    axes[1, col].legend(fontsize=9)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Error distribution ---
plt.figure(figsize=(8, 4))
plt.hist(rel_l2_errors, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(np.mean(rel_l2_errors), color='red', linestyle='--', linewidth=2,
            label=f'Mean = {np.mean(rel_l2_errors):.2e}')
plt.xlabel('Relative $L^2$ Error')
plt.ylabel('Count')
plt.title('Error Distribution Over Test Set')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Generalization: Test with a New Function

Let's test the trained DeepONet on a **hand-crafted function** that was never in the training data to see how well it generalizes.

In [ ]:
# Define a custom function with specific coefficients
a1_c, w1_c, a2_c, w2_c = 1.5, 2.0, -1.0, 5.0

# Evaluate u directly at sensor locations
u_custom = a1_c * np.sin(w1_c * np.pi * x_sensors) + a2_c * np.sin(w2_c * np.pi * x_sensors)

# Analytical s at query locations
s_custom_true = (a1_c / (w1_c * np.pi) * (1 - np.cos(w1_c * np.pi * y_query_np)) +
                 a2_c / (w2_c * np.pi) * (1 - np.cos(w2_c * np.pi * y_query_np)))

# DeepONet prediction
u_custom_tensor = torch.tensor(u_custom.reshape(1, -1), dtype=torch.float32).to(device)
with torch.no_grad():
    s_custom_pred = model(u_custom_tensor, y_query).cpu().numpy().flatten()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_sensors, u_custom, color='steelblue', linewidth=2)
axes[0].set_title(f'$u(x) = {a1_c:+.1f}\\sin({w1_c:.0f}\\pi x) {a2_c:+.1f}\\sin({w2_c:.0f}\\pi x)$', fontsize=12)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$u(x)$')
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_query_np, s_custom_true, 'k-', linewidth=2, label='Analytical')
axes[1].plot(y_query_np, s_custom_pred, 'r--', linewidth=2, label='DeepONet')
rel_err = np.linalg.norm(s_custom_pred - s_custom_true) / np.linalg.norm(s_custom_true)
axes[1].set_title(f'Anti-derivative — Rel. $L^2$ error: {rel_err:.2e}', fontsize=12)
axes[1].set_xlabel('$y$')
axes[1].set_ylabel('$s(y)$')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **DeepONet learned the anti-derivative operator** — not just one function, but the mapping from *any* input function to its integral.

2. **Once trained, inference is instant** — evaluating the anti-derivative for a new function takes a single forward pass (milliseconds).

3. **The architecture is modular** — the branch encodes *what* function we have, the trunk encodes *where* we evaluate. They combine via a dot product.

4. **Generalization** — the model can handle functions it has never seen during training, as long as they are "similar enough" to the training distribution (GRF).